In [1]:
!pip install numpy pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [70]:
import pandas as pd
import matplotlib.pyplot as plt

def main():
    print("Loading data from pool_allocator_lob_latencies.csv...")
    try:
        df = pd.read_csv('pool_allocator_lob_latencies.csv')
    except FileNotFoundError:
        print("Error: lob_latencies.csv not found in the current directory.")
        return

    operations = df['operation'].unique()
    
    print("\n--- Latency Percentiles (nanoseconds) ---")
    percentiles = [0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 0.9999]
    perc_stats = df.groupby('operation')['nanoseconds'].quantile(percentiles).unstack()
    perc_stats.columns = [f"{p*100:g}th" for p in percentiles]
    print(perc_stats.to_string(float_format="%.2f"))
    print("-----------------------------------------\n")

    # 1. Percentile Plot
    plt.figure(figsize=(10, 6))
    perc_stats.T.plot(kind='bar', colormap='Set1', ax=plt.gca())
    plt.title('Latency Percentiles by Operation')
    plt.ylabel('Latency (ns)')
    plt.xlabel('Percentile')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_1_percentiles.png', dpi=300)
    plt.close()
    
    # 2. Sequential Time Series
    plt.figure(figsize=(12, 6))
    window_size = 1000 
    for op in operations:
        op_df = df[df['operation'] == op].copy()
        op_df = op_df.reset_index(names='seq_index') 
        op_df['rolling_ns'] = op_df['nanoseconds'].rolling(window=window_size, min_periods=1).mean()
        plt.plot(op_df['seq_index'], op_df['rolling_ns'], label=f'{op} (Rolling {window_size})', alpha=0.8)

    plt.title('Latency over Time (Sequential Trend)')
    plt.ylabel('Average Latency (ns)')
    plt.xlabel('Operation Sequence')
    plt.legend()
    plt.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_2_sequential.png', dpi=300)
    plt.close()

    # 3. Standard Histogram (Linear)
    plt.figure(figsize=(10, 6))
    for op in operations:
        data = df[df['operation'] == op]['nanoseconds']
        clip_val = data.quantile(0.999) 
        plt.hist(data[data < clip_val], bins=100, alpha=0.5, label=op)
        
    plt.title('Latency Distribution (Linear) - Clipped at 99.9th %')
    plt.ylabel('Frequency')
    plt.xlabel('Latency (ns)')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_3_histogram_linear.png', dpi=300)
    plt.close()

    # 4. Log-Y Histogram
    plt.figure(figsize=(10, 6))
    for op in operations:
        data = df[df['operation'] == op]['nanoseconds']
        clip_val = data.quantile(0.999) 
        plt.hist(data[data < clip_val], bins=100, alpha=0.5, label=op)
        
    plt.title('Latency Distribution (Log-Y) - Clipped at 99.9th %')
    plt.ylabel('Frequency (Log Scale)')
    plt.xlabel('Latency (ns)')
    plt.yscale('log')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_4_histogram_log.png', dpi=300)
    plt.close()

    # 5. Hash Map Size vs Capacity
    plt.figure(figsize=(12, 6))
    downsampled_df = df.iloc[::1000] 
    
    plt.plot(downsampled_df.index, downsampled_df['map_size'], label='Map Size (Active Orders)', color='blue', linewidth=2)
    plt.plot(downsampled_df.index, downsampled_df['map_capacity'], label='Map Capacity (Allocated Slots)', color='orange', linestyle='--', linewidth=2)
    plt.title('SwissTable Hash Map Memory State')
    plt.ylabel('Number of Elements / Slots')
    plt.xlabel('Operation Sequence')
    plt.legend()
    plt.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_5_map_state.png', dpi=300)
    plt.close()

    # 6. Hash Map Load Factor
    plt.figure(figsize=(12, 6))
    plt.plot(downsampled_df.index, downsampled_df['load_factor'], label='Load Factor', color='green', linewidth=2)
    plt.axhline(y=0.875, color='red', linestyle=':', label='Rehash Threshold (~0.875)')
    
    plt.title('SwissTable Load Factor')
    plt.ylabel('Load Factor Ratio')
    plt.xlabel('Operation Sequence')
    plt.legend()
    plt.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('plot_6_load_factor.png', dpi=300)
    plt.close()

    print("Successfully generated 6 separate image files in the current directory.")

if __name__ == '__main__':
    main()

Loading data from pool_allocator_lob_latencies.csv...

--- Latency Percentiles (nanoseconds) ---
            50th   75th   90th   95th   99th  99.9th  99.99th
operation                                                    
ADD        69.71  95.36 151.45 187.91 268.84  437.51  5329.54
CANCEL    173.09 250.81 322.93 359.79 461.56  663.50 17179.72
-----------------------------------------

Successfully generated 6 separate image files in the current directory.
